<a href="https://colab.research.google.com/github/zombimann/Mathematical-video-animations-and-visualization/blob/main/Canny_Edge_Extraction_Demonstration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Canny Edge Detection Visualization Project

This notebook generates a high-quality, branded video visualization of the **Canny Edge Detection** algorithm. It is designed for public outreach and educational purposes, demonstrating how computer vision "sees" structures in images.

## Theoretical Overview: How Canny Works

Developed by John F. Canny in 1986, the Canny edge detector is a multi-stage algorithm used to detect a wide range of edges in images. The process follows these key steps:

1.  **Noise Reduction**: Since edge detection is susceptible to noise in the image, the first step is to remove noise with a 5x5 Gaussian filter.
2.  **Finding Intensity Gradients**: The smoothed image is filtered with a Sobel kernel in both horizontal and vertical directions to get the first derivative in both directions ($G_x$ and $G_y$). From these, we find the edge gradient and direction for each pixel.
3.  **Non-maximum Suppression**: After getting gradient magnitude and direction, a full scan of the image is done to remove any unwanted pixels which may not constitute the edge. For this, at every pixel, pixel is checked if it is a local maximum in its neighborhood in the direction of gradient.
4.  **Hysteresis Thresholding**: This stage decides which are all edges are actually edges and which are not. For this, we need two threshold values, `minVal` and `maxVal`. Any edges with intensity gradient more than `maxVal` are sure to be edges and those below `minVal` are sure to be non-edges, so discarded. Those who lie between these two thresholds are classified edges or non-edges based on their connectivity.

## Script Features
*   **Parametric Design**: Easily adjust frames per second, scene duration, and sensitivity.
*   **Branded Output**: Includes custom watermarks and a professional title card.
*   **Automated Workflow**: Fetches assets from Unsplash and handles video rendering and download automatically.

In [2]:
!pip install opencv-python-headless moviepy

In [ ]:
"""
Canny Edge Detection Visualization Script
Author: Mugambi Ndwiga (@craftsandengineering)

This script generates a 15-second video demonstrating the Canny edge detection algorithm.
It fetches high-quality images, processes them with different sensitivity thresholds,
and outputs a branded comparison video with a closing title card.
"""

import cv2
import numpy as np
import requests
from PIL import Image
from io import BytesIO
from moviepy.editor import ImageSequenceClip
from google.colab import files
import os

# --- CONFIGURATION PARAMETERS ---
PARAMS = {
    'video_fps': 24,
    'duration_per_scene': 2, # Seconds per image
    'total_scenes': 5,
    'output_filename': 'canny_outreach_viz.mp4',
    'author': 'Mugambi Ndwiga',
    'instagram': '@craftsandengineering',
    'video_title': 'CANNY EDGE DETECTION',
    'canny_configs': [
        {'low': 15, 'high': 45, 'label': 'High Sensitivity (Detail)'},
        {'low': 250, 'high': 500, 'label': 'Low Sensitivity (Structure)'}
    ],
    'image_urls': [
        'https://images.unsplash.com/photo-1541963463532-d68292c34b19?w=1280',
        'https://images.unsplash.com/photo-1441974231531-c6227db76b6e?w=1280',
        'https://images.unsplash.com/photo-1501785888041-af3ef285b470?w=1280',
        'https://images.unsplash.com/photo-1470071459604-3b5ec3a7fe05?w=1280',
        'https://images.unsplash.com/photo-1447752875215-b2761acb3c5d?w=1280'
    ]
}

def download_image(url):
    """Downloads an image from a URL and returns a NumPy array."""
    response = requests.get(url)
    img = Image.open(BytesIO(response.content)).convert('RGB')
    return np.array(img)

def process_frame(img, author, handle):
    """Processes a single image into a three-panel branded frame."""
    target_h, target_w = 720, 1280
    panel_w = target_w // 3

    # Resize and convert to grayscale for processing
    orig_panel = cv2.resize(img, (panel_w, target_h))
    gray = cv2.cvtColor(orig_panel, cv2.COLOR_RGB2GRAY)

    # Generate Canny edge variations
    c1 = cv2.Canny(gray, PARAMS['canny_configs'][0]['low'], PARAMS['canny_configs'][0]['high'])
    c1_rgb = cv2.cvtColor(c1, cv2.COLOR_GRAY2RGB)

    c2 = cv2.Canny(gray, PARAMS['canny_configs'][1]['low'], PARAMS['canny_configs'][1]['high'])
    c2_rgb = cv2.cvtColor(c2, cv2.COLOR_GRAY2RGB)

    # Stack panels horizontally
    combined = np.hstack([orig_panel, c1_rgb, c2_rgb])
    combined = cv2.resize(combined, (target_w, target_h))

    # Add Persistent Video Title (Top Center)
    cv2.putText(combined, PARAMS['video_title'], (420, 40),
                cv2.FONT_HERSHEY_DUPLEX, 1.0, (255, 255, 255), 2, cv2.LINE_AA)

    # Add Author Branding (Bottom Left)
    cv2.putText(combined, f"{author} | {handle}", (20, target_h - 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 1, cv2.LINE_AA)

    # Add Panel Labels
    cv2.putText(combined, "Original", (20, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
    cv2.putText(combined, PARAMS['canny_configs'][0]['label'], (panel_w + 20, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    cv2.putText(combined, PARAMS['canny_configs'][1]['label'], (2*panel_w + 20, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 180, 255), 2)

    return combined

print("Step 1: Processing image scenes...")
video_frames = []
for i in range(PARAMS['total_scenes']):
    img_raw = download_image(PARAMS['image_urls'][i])
    frame = process_frame(img_raw, PARAMS['author'], PARAMS['instagram'])
    # Repeat frame for the duration of the scene
    for _ in range(PARAMS['video_fps'] * PARAMS['duration_per_scene']):
        video_frames.append(frame)

print("Step 2: Creating final title card...")
title_card = np.zeros((720, 1280, 3), dtype=np.uint8)
cv2.putText(title_card, "Dramatic Edge Detection", (300, 300), cv2.FONT_HERSHEY_TRIPLEX, 2, (255, 255, 255), 3)
cv2.putText(title_card, f"Created by: {PARAMS['author']}", (420, 400), cv2.FONT_HERSHEY_SIMPLEX, 1, (200, 200, 200), 2)
cv2.putText(title_card, f"Follow: {PARAMS['instagram']}", (420, 460), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 180, 255), 2)

# Append 5-second end card
for _ in range(PARAMS['video_fps'] * 5):
    video_frames.append(title_card)

print("Step 3: Rendering video file...")
clip = ImageSequenceClip(video_frames, fps=PARAMS['video_fps'])
clip.write_videofile(PARAMS['output_filename'], codec='libx264', bitrate='12000k')

print("Rendering complete!")
from IPython.display import Video
display(Video(PARAMS['output_filename'], embed=True))
files.download(PARAMS['output_filename'])
